# BTS Digital Twin (NVS) — Đóng gói submission từ checkpoint có sẵn (Round 2)

Notebook này **KHÔNG train** — chỉ tải 7 checkpoint (`gs_model/`) của 7 scene
round 2 (5 scene BTS + `bonsai` + `chair`, xem `pipeline/common/scenes.py`) đã train ở
`kaggle_private.ipynb` (đã upload lên Google Drive), render lại ảnh tại các pose test
rồi đóng gói `submission.zip`. Chạy nhanh (không tốn GPU-giờ cho train, chỉ tốn cho
render — vài phút/scene).

**Trước khi chạy, cần điền:**
1. Settings → Accelerator: **GPU T4 x2** (hoặc P100) → Internet: **On**.
2. `REPO_URL` ở Bước 3, `GDRIVE_URL` ở Bước 4 (dataset — cần `test_poses.csv` từng scene).
3. `CHECKPOINT_LINKS` ở Bước 6 — dán đủ 7 link Google Drive (mỗi scene 1 link **file**
   `point_cloud.ply` đã upload theo hướng dẫn ở Bước 7 của `kaggle_private.ipynb`).

**Bảo mật:** để notebook này **Private**.

## Bước 1 — Cài đặt

In [ ]:
import torch, subprocess, sys
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG có GPU — vào Settings bật Accelerator GPU trước khi chạy tiếp")
print("Torch:", torch.__version__, "| CUDA build:", torch.version.cuda)

In [ ]:
!pip install -q pycolmap "scikit-image>=0.19" lpips plyfile tqdm gdown

## Bước 2 — Clone + build 3D Gaussian Splatting

Repo gốc `graphdeco-inria/gaussian-splatting` — dùng để train/render, không tự viết lại
trainer (quá nhiều chi tiết dễ sai: densification, SH coefficients...). Bước build
2 CUDA extension (`diff-gaussian-rasterization`, `simple-knn`) mất khoảng 2-5 phút.

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git
!pip install -q ./gaussian-splatting/submodules/diff-gaussian-rasterization
!pip install -q ./gaussian-splatting/submodules/simple-knn

import os
os.environ["GS_REPO"] = "/kaggle/working/gaussian-splatting"
print("GS_REPO =", os.environ["GS_REPO"])

## Bước 3 — Lấy code pipeline từ Git repo của bạn (khuyến nghị để **Private**)

Repo Private vẫn clone được bình thường trên Kaggle, chỉ cần xác thực bằng
**Personal Access Token (PAT)** thay vì mật khẩu. Các bước 1 lần:

1. Đẩy code lên GitHub, chọn **Private** khi tạo repo (không ai ngoài bạn xem được,
   kể cả khi bạn share notebook Kaggle này cho người khác sau này).
2. Tạo token: GitHub → **Settings → Developer settings → Personal access tokens →
   Fine-grained tokens → Generate new token**. Chọn:
   - Repository access: **Only select repositories** → chọn đúng repo vừa tạo.
   - Permissions → Contents: **Read-only** (không cần quyền gì khác).
   - Đặt ngày hết hạn (Expiration) ngắn thôi, vd 30-90 ngày — hết hạn thì tạo token mới.
3. **Copy token, dán vào Kaggle Secrets (KHÔNG dán thẳng vào code)**: trong notebook
   Kaggle, vào menu **Add-ons → Secrets → Add a new secret** → Label đặt đúng tên
   `GITHUB_TOKEN`, Value dán token vừa copy → Save. Cell bên dưới sẽ tự đọc secret
   này lúc chạy, token không hề xuất hiện trong code/notebook — kể cả nếu lỡ share
   notebook cho người khác, họ cũng không nhìn thấy được token của bạn.

Nếu push CẢ project (gồm `Đề bài.md`, `Hướng đi.md`, `Dataset/`, `pipeline/`...)
làm 1 repo cũng được — cell dưới tự dò tìm thư mục con tên `pipeline` (chứa `common/`
và `scripts/`) ở bất kỳ độ sâu nào trong repo, không cần đúng ngay gốc repo.

In [ ]:
REPO_URL = "https://github.com/ThongLuc2k3/BTS-Digital-Twin.git"

# GITHUB_TOKEN: ưu tiên lấy từ Kaggle Secrets (an toàn, không lộ trong code).
# Chỉ cần dán thẳng vào biến bên dưới nếu bạn KHÔNG dùng Kaggle Secrets (kém an
# toàn hơn — token sẽ nằm lộ trong notebook, đừng share notebook cho ai nếu làm vậy).
GITHUB_TOKEN = ""

try:
    from kaggle_secrets import UserSecretsClient
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
        print("Đã lấy GITHUB_TOKEN từ Kaggle Secrets.")
except Exception:
    if not GITHUB_TOKEN:
        print("Không tìm thấy Kaggle Secret 'GITHUB_TOKEN' (bỏ qua nếu repo Public, "
              "hoặc bạn đã dán token thẳng vào biến GITHUB_TOKEN ở trên).")

assert REPO_URL, "Chưa điền REPO_URL — dán link git repo chứa thư mục pipeline/ vào biến này rồi chạy lại cell."

clone_url = REPO_URL
if GITHUB_TOKEN and "github.com" in REPO_URL:
    clone_url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@")

!rm -rf /kaggle/working/_repo_clone
!git clone --depth 1 "{clone_url}" /kaggle/working/_repo_clone

In [ ]:
# Tự dò thư mục "pipeline" (chứa common/ và scripts/) ở bất kỳ đâu trong repo vừa
# clone, rồi symlink về /kaggle/working/pipeline — mọi cell sau đều gọi script từ đây.
import os
import shutil
from pathlib import Path

CLONE_ROOT = Path("/kaggle/working/_repo_clone")
found = None
for dirpath, dirnames, filenames in os.walk(CLONE_ROOT):
    p = Path(dirpath)
    if p.name == "pipeline" and "common" in dirnames and "scripts" in dirnames:
        found = p
        break
if found is None and (CLONE_ROOT / "common").exists() and (CLONE_ROOT / "scripts").exists():
    found = CLONE_ROOT  # trường hợp bạn push thẳng NỘI DUNG pipeline/ làm gốc repo

if found is None:
    raise SystemExit(
        "Không tìm thấy thư mục 'pipeline' (chứa common/ và scripts/) trong repo vừa clone.\n"
        f"Nội dung clone nằm ở {CLONE_ROOT} — kiểm tra lại đã push đúng thư mục pipeline/ lên git chưa."
    )

print("Tìm thấy code pipeline tại:", found)
target = Path("/kaggle/working/pipeline")
if target.is_symlink():
    target.unlink()
elif target.exists():
    shutil.rmtree(target)
os.symlink(found.resolve(), target)
print("Đã symlink -> /kaggle/working/pipeline ->", found.resolve())

Path("/kaggle/working/pipeline/work").mkdir(parents=True, exist_ok=True)

## Bước 4 — Tải dataset từ Google Drive

Điền link chia sẻ Google Drive (chế độ "Anyone with the link") vào `GDRIVE_URL` bên
dưới — file phải là **1 file .zip** chứa `Dataset/VAI_NVS_DATA_ROUND2/<scene>/...`
(zip nguyên thư mục `Dataset` như trong repo, hoặc chỉ riêng `VAI_NVS_DATA_ROUND2`
cũng được — cell dưới tự dò tìm thư mục `VAI_NVS_DATA_ROUND2` ở bất kỳ độ sâu nào
trong zip).

In [ ]:
GDRIVE_URL = "https://drive.google.com/file/d/178EL7jCSVD59q19SMpeOgnOfOIC66I_t/view?usp=drive_link"

assert GDRIVE_URL, "Chưa điền GDRIVE_URL — dán link chia sẻ Google Drive (Anyone with the link) của file zip dataset vào biến này rồi chạy lại cell."

import os
os.makedirs("/kaggle/working/_dataset_raw", exist_ok=True)
!gdown --fuzzy "{GDRIVE_URL}" -O /kaggle/working/dataset.zip
!unzip -q -o /kaggle/working/dataset.zip -d /kaggle/working/_dataset_raw
print("Đã giải nén xong, đang dò tìm thư mục dataset ...")

In [ ]:
# Tự dò thư mục chứa các scene round 2 phẳng (HCM0421/, chair/, bonsai/...) ở bất
# kỳ đâu trong zip vừa giải nén, rồi symlink về đúng vị trí mà
# pipeline/common/scenes.py cần: /kaggle/working/Dataset/VAI_NVS_DATA_ROUND2
#
# KHÔNG bắt buộc thư mục bọc ngoài phải tên đúng "VAI_NVS_DATA_ROUND2" — chỉ cần
# TÌM ĐƯỢC 1 thư mục (kể cả chính gốc giải nén, nếu zip không có lớp bọc ngoài)
# chứa đủ NHIỀU scene mong đợi trực tiếp bên trong (đồng bộ với fix đã áp trong
# kaggle_private.ipynb — cùng 1 dataset zip nên cùng rủi ro cấu trúc thư mục).
#
# Danh sách tên scene lặp lại thủ công ở đây (không import common.scenes) vì
# sys.path chưa trỏ tới pipeline/ ở bước này (việc đó làm ở cell kiểm tra ngay
# sau) — giữ đồng bộ với BTS_SCENES/GENERIC_SCENES trong pipeline/common/scenes.py
# nếu sau này thêm/bớt scene.
import os
from pathlib import Path

_expected_scene_dirs = {"HCM0421", "HCM0539", "HCM0540", "HCM0644", "HCM0674", "bonsai", "chair"}
_MIN_MATCH = 4  # đủ scene trùng khớp để tin đây đúng là thư mục dataset (tránh khớp nhầm thư mục rác)

RAW_ROOT = Path("/kaggle/working/_dataset_raw")
found = None
best_match = 0
for dirpath, dirnames, filenames in os.walk(RAW_ROOT):
    # __MACOSX/ là rác do Mac tạo khi nén zip — nó TỰ NHÂN BẢN y hệt cấu trúc thư
    # mục thật (HCM0421/, train/images/...) nhưng file bên trong chỉ là file
    # rác metadata "._<tên file>", không phải dữ liệu thật. Phải loại trừ, nếu
    # không os.walk có thể tìm trúng "__MACOSX/..." trước bản thật.
    dirnames[:] = [d for d in dirnames if d != "__MACOSX" and not d.startswith(".")]
    n_match = len(_expected_scene_dirs & set(dirnames))
    if n_match > best_match:
        best_match = n_match
        found = Path(dirpath)
    if n_match == len(_expected_scene_dirs):
        break  # khớp đủ cả 7 — dừng sớm, khỏi walk tiếp cho nhanh

if found is None or best_match < _MIN_MATCH:
    raise SystemExit(
        "Không tìm thấy thư mục nào chứa đủ scene round 2 (HCM0421, chair, bonsai...) "
        f"trong zip vừa giải nén (khớp nhiều nhất: {best_match}/7, cần >= {_MIN_MATCH}).\n"
        f"Nội dung giải nén nằm ở {RAW_ROOT} — chạy `!find {RAW_ROOT} -maxdepth 3` ở 1 cell "
        "khác để xem cấu trúc thật, đối chiếu lại với file zip đã upload lên Google Drive."
    )

print(f"Tìm thấy ({best_match}/7 scene khớp):", found)
target = Path("/kaggle/working/Dataset/VAI_NVS_DATA_ROUND2")
target.parent.mkdir(parents=True, exist_ok=True)
if target.exists() or target.is_symlink():
    target.unlink() if target.is_symlink() else None
if not target.exists():
    os.symlink(found.resolve(), target)
print("Đã symlink ->", target, "->", found.resolve())

# QUAN TRỌNG: code (git clone) và dataset (Google Drive) không nằm chung 1 thư mục
# gốc trên Kaggle như lúc chạy local, nên common/scenes.py KHÔNG thể tự suy ra
# đường dẫn dataset bằng "đi lên N cấp từ vị trí file code" — phải khai báo thẳng
# qua biến môi trường này (đọc bởi pipeline/common/scenes.py).
os.environ["BTS_DATASET_ROOT"] = str(target.resolve())
print("Đã set BTS_DATASET_ROOT =", os.environ["BTS_DATASET_ROOT"])

In [ ]:
# Kiểm tra lại: liệt kê đủ 7 scene round 2 + scene nào có sparse hợp lệ — dataset
# đầy đủ thì kỳ vọng has_valid_provided_sparse=True cho CẢ 7 scene.
# Nếu train_ok=False hết cho mọi scene, kiểm tra lại giá trị BTS_DATASET_ROOT ở cell
# trên có trỏ đúng chỗ chứa VAI_NVS_DATA_ROUND2 hay không (thường do dataset.zip
# tải/giải nén thiếu — thử xoá /kaggle/working/_dataset_raw và tải lại từ đầu).
import sys
sys.path.insert(0, "/kaggle/working/pipeline")
from common.scenes import all_scenes, DATASET_ROOT

print("DATASET_ROOT =", DATASET_ROOT, "| tồn tại:", DATASET_ROOT.exists())
for s in all_scenes():
    ok_train = s.train_images_dir.exists()
    ok_csv = s.test_poses_csv.exists()
    n_train = len(list(s.train_images_dir.glob("*"))) if ok_train else 0
    print(f"{s.name:10s} {s.domain:8s} train_ok={ok_train} n_train={n_train:4d} "
          f"csv_ok={ok_csv} has_valid_provided_sparse={s.has_valid_provided_sparse()}")

## Bước 5 — Tải checkpoint từ Google Drive cho cả 7 scene

Điền đủ 7 link Google Drive (mỗi scene 1 link **THƯ MỤC `gs_model`** — đúng thư mục mà
`kaggle_private.ipynb` (Bước 7) hướng dẫn tải nguyên vẹn lên Drive, KHÔNG phải chỉ link
1 file `point_cloud.ply` đơn lẻ) vào dict `CHECKPOINT_LINKS` bên dưới. Nhớ share thư mục
đó ở chế độ "Anyone with the link".

**LƯU Ý QUAN TRỌNG (bug thật đã sửa):** bản trước của cell này chỉ tải mỗi file
`point_cloud.ply` — thiếu `cfg_args` và `pipeline_train_flags.json` nằm cùng cấp trong
`gs_model/`. Thiếu 2 file đó, `04_render_test_poses.py` KHÔNG có cách nào biết lúc train
thật sự có bật `--antialiasing` hay không — nó âm thầm mặc định `antialiasing=False`
(chỉ in cảnh báo, không dừng lại), làm SAI HOÀN TOÀN ảnh render nộp bài cho MỌI scene
(cả 7 scene đều train với `ANTIALIASING=1`) mà không có lỗi rõ ràng nào báo ra — đúng
kiểu bug round 1 mà `10_sanity_check_render.py` được viết ra để bắt. Cell dưới giờ tải
NGUYÊN thư mục `gs_model` (dùng `gdown --folder`) và tự kiểm tra đủ `cfg_args` +
`pipeline_train_flags.json` trước khi cho qua — báo lỗi rõ ràng nếu thiếu thay vì âm
thầm render sai.

In [ ]:
CHECKPOINT_LINKS = {
    "HCM0421": "",
    "HCM0539": "",
    "HCM0540": "",
    "HCM0644": "",
    "HCM0674": "",
    "bonsai": "",
    "chair": "",
}

missing = [s for s, link in CHECKPOINT_LINKS.items() if not link]
assert not missing, f"Chưa điền link Google Drive cho scene: {missing}"

In [ ]:
import shutil
from pathlib import Path

for scene, link in CHECKPOINT_LINKS.items():
    dest_dir = Path(f"/kaggle/working/pipeline/work/{scene}")
    dest_dir.mkdir(parents=True, exist_ok=True)
    gs_model_dst = dest_dir / "gs_model"
    shutil.rmtree(gs_model_dst, ignore_errors=True)

    raw_dl_dir = Path(f"/kaggle/working/_ckpt_raw/{scene}")
    shutil.rmtree(raw_dl_dir, ignore_errors=True)
    raw_dl_dir.mkdir(parents=True, exist_ok=True)
    print(f"===== {scene}: tải thư mục gs_model từ Drive =====")
    !gdown --fuzzy --folder "{link}" -O "{raw_dl_dir}"

    # gdown --folder có thể tự tạo thêm 1 lớp thư mục con trùng tên folder Drive gốc
    # (tuỳ version/cấu trúc) — tự dò lớp chứa "cfg_args" (file luôn nằm ở gốc
    # gs_model/, do train.py tự ghi) thay vì giả định cứng độ sâu, cùng cách làm
    # với các cell tự dò dataset ở trên (tránh lệ thuộc cấu trúc zip/folder chính xác).
    candidates = [p.parent for p in raw_dl_dir.rglob("cfg_args")]
    assert candidates, (
        f"{scene}: tải xong nhưng KHÔNG tìm thấy file 'cfg_args' trong {raw_dl_dir} — kiểm tra lại "
        f"link Drive có đúng là THƯ MỤC gs_model/ (không phải chỉ mỗi point_cloud.ply) và đã share "
        f"\"Anyone with the link\" chưa.")
    src_root = candidates[0]

    assert (src_root / "pipeline_train_flags.json").exists(), (
        f"{scene}: thiếu pipeline_train_flags.json trong {src_root} — thư mục gs_model tải lên Drive "
        f"phải nguyên vẹn (không tự xoá bớt file con nào), nếu không render sẽ SAI antialiasing (âm "
        f"thầm, không báo lỗi rõ — xem docstring 04_render_test_poses.py::read_pipeline_train_flags).")
    n_ply = len(list(src_root.glob("point_cloud/iteration_*/point_cloud.ply")))
    assert n_ply > 0, f"{scene}: không tìm thấy point_cloud.ply nào trong {src_root}/point_cloud/iteration_*/"

    shutil.copytree(src_root, gs_model_dst)
    shutil.rmtree(raw_dl_dir, ignore_errors=True)
    print(f"-> OK, {gs_model_dst} ({n_ply} checkpoint iteration, cfg_args + pipeline_train_flags.json có mặt)")

## Bước 6 — Render lại ảnh test cho cả 7 scene

Dùng đúng checkpoint vừa tải (iteration lớn nhất có sẵn cho mỗi scene, mặc định của
`04_render_test_poses.py`).

In [ ]:
for scene in CHECKPOINT_LINKS:
    !python /kaggle/working/pipeline/scripts/04_render_test_poses.py --scene {scene}

## Bước 7 — Đóng gói + kiểm tra `submission.zip`

Script tự kiểm tra đủ 7 scene / đủ ảnh / đúng kích thước trước khi nén, và verify
lại chính file zip vừa tạo (xem `plan.md` mục 8 — checklist trước khi nộp).

**Lưu ý `--jpeg_quality 98`**: đã đo trên máy local (dữ liệu round 1), quality=98 cho
tổng ~304.1MB — dư an toàn ~46MB (~13%) so với hạn 350MB. Round 2 có thêm 2 scene
`bonsai`/`chair` độ phân giải khác (1920×1080, 720×1280) nên dung lượng thực tế có thể
lệch so với con số đo trên round 1 — cell dưới tự in dung lượng zip và assert lỗi ngay
nếu vượt 350MB (xác nhận lại giới hạn 350MB này còn đúng cho round 2 hay không trước
khi tin tưởng hoàn toàn). Chất lượng ảnh giữa 95/98 gần như không khác biệt (PSNR
~50dB, chênh lệch <1/255 mức xám so với gần-lossless).

In [ ]:
!python /kaggle/working/pipeline/scripts/06_package_submission.py \
    --out /kaggle/working/submission.zip \
    --filename_mode literal \
    --jpeg_quality 98

import os
size_mb = os.path.getsize('/kaggle/working/submission.zip') / 1e6
print(f"Dung luong zip: {size_mb:.1f} MB (gioi han 350MB, quality=98 du an toan ~46MB tren round 1)")
assert size_mb < 350, f"VUOT GIOI HAN 350MB ({size_mb:.1f}MB) — giam --jpeg_quality xuong (vd 95) roi chay lai cell nay."

## Bước 8 — Lưu kết quả

`submission.zip` đã nằm ở `/kaggle/working/` — bấm **Save Version** (góc
trên phải) để Kaggle giữ lại file này trong tab "Output" của notebook, tải về từ đó
để nộp bài.